# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a guide for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL at:
`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL for the dataset
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Access metadata fields
metadata = dataset.metadata
print(f"Dataset Title: {metadata.name}\n")
print(f"Description: {metadata.description}\n")

## 2. Data Overview
Review available record sets (`cr:RecordSet`), their fields (`cr:Field`), and corresponding `@id`s.

In Croissant, each entity such as record sets and fields is uniquely identified by an `@id`. We'll enumerate the available record sets and examine their fields, all referenced by their `@id`.

In [ ]:
# Helper to pretty print
from pprint import pprint

# List all available record sets
record_sets = metadata.record_sets
print(f"Found {len(record_sets)} record sets:")
for idx, record_set in enumerate(record_sets):
    print(f"[{idx}] RecordSet @id: {record_set.id} | name: {record_set.name if hasattr(record_set, 'name') else '(no name)'}")

# Show fields and columns for each RecordSet
for record_set in record_sets:
    print(f"\nRecordSet @id: {record_set.id}")
    if hasattr(record_set, 'fields') and record_set.fields:
        print(f"  Fields:")
        for field in record_set.fields:
            print(f"    - Field @id: {field.id}" + (f" | name: {field.name}" if hasattr(field, 'name') else ''))
            # List the columns for the field (if any)
            if hasattr(field, 'columns') and field.columns:
                for column in field.columns:
                    print(f"        - Column @id: {column.id}" + (f" | name: {column.name}" if hasattr(column, 'name') else ''))
    else:
        print("  No fields found.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame. Use the record set and field `@id`s from the overview above.

We'll demonstrate extraction using the first available record set. You can update the `primary_record_set_id` variable to another `@id` as needed.


In [ ]:
# Extract data from each record set as DataFrames

dataframes = {}
all_record_set_ids = [rs.id for rs in record_sets]

# For demo, use the first record set for main analysis
primary_record_set_id = all_record_set_ids[0]

print(f"Loading records for RecordSet @id: {primary_record_set_id}\n")
for record_set_id in all_record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)
    print(f"Loaded DataFrame for @id: {record_set_id} with shape: {dataframes[record_set_id].shape}")

# Display field names/columns for the primary record set
print(f"\nColumns for primary record set (@id: {primary_record_set_id}):")
print(dataframes[primary_record_set_id].columns.tolist())
dataframes[primary_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply data processing to the extracted DataFrame. Here, we:
- select a numeric field (referenced by `@id` as shown above),
- filter records,
- normalize a numeric field,
- group by a categorical field by `@id`.

**Update the variable assignments below for your desired fields using the `@id`s from the overview.**

In [ ]:
# --- Customize these @id assignments to analyze different fields ---
record_set_id = primary_record_set_id
df = dataframes[record_set_id]

# Example: choose a numeric field by its @id (update as appropriate)
# For demonstration, we'll use the first numeric-looking column. Adjust as needed.

numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
if numeric_fields:
    numeric_field_id = numeric_fields[0]
    print(f"Using numeric field for analysis (@id): {numeric_field_id}")
else:
    raise ValueError("No numeric fields available in the DataFrame; cannot proceed with EDA.")

threshold = df[numeric_field_id].mean()  # Example: use mean for filtering
filtered_df = df[df[numeric_field_id] > threshold].copy()
print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (kept {len(filtered_df)} records)")
display(filtered_df.head())

# Add a normalized column
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id} for filtered records:")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by a field -- use the first non-numeric column (usually categorical/text)
potential_group_fields = [col for col in df.columns if not pd.api.types.is_numeric_dtype(df[col])]
group_field_id = None
for field in potential_group_fields:
    # Grouping only if non-unique enough (<90% unique)
    if df[field].nunique() < 0.9 * len(df):
        group_field_id = field
        break

if group_field_id:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"\nMean {numeric_field_id} per group by {group_field_id}:")
    display(grouped_df.head())
else:
    print("No suitable categorical field found for grouping.")

## 5. Visualization
Visualize the distribution of the selected numeric field and its normalized values. If grouping was possible, plot the grouped means by group field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot distribution of the numeric field
plt.figure(figsize=(8,4))
sns.histplot(filtered_df[numeric_field_id], kde=True, color='skyblue')
plt.title(f"Distribution of {numeric_field_id} (filtered)")
plt.xlabel(numeric_field_id)
plt.ylabel("Frequency")
plt.show()

# Plot normalized values
plt.figure(figsize=(8,4))
sns.histplot(filtered_df[f"{numeric_field_id}_normalized"], kde=True, color='orange')
plt.title(f"Normalized {numeric_field_id} Distribution")
plt.xlabel(f"{numeric_field_id}_normalized")
plt.ylabel("Frequency")
plt.show()

# If grouping was done, plot grouped means
if group_field_id and 'grouped_df' in locals():
    plt.figure(figsize=(10,4))
    sns.barplot(data=grouped_df, x=group_field_id, y=numeric_field_id)
    plt.title(f"Mean {numeric_field_id} by {group_field_id}")
    plt.xticks(rotation=60)
    plt.show()

## 6. Conclusion
We successfully loaded the FAIR² dataset metadata and records, explored its structure using Croissant `@id` identifiers, and performed basic data processing and visualization with `mlcroissant`. This workflow can be readily extended by referencing other record sets and columns using their explicit `@id` values as shown in Section 2.

Please consult the [mlcroissant documentation](https://mlcommons.github.io/croissant/) and the [dataset Croissant schema](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) for further fields, entities, and advanced usage.